# Data Analysis
This notebook explores the data generated with the `scripts/build_datasets.py` script. That dataset is dependent on the batch of replays used to create it, so results may vary from run to run.

## Setup

In [1]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from starcraft_predictor.replays.unit_tracker import TRACKED_UNIT_TYPES, TRACKED_UPGRADE_TYPES
from starcraft_predictor.replays.player_stats_tracker import PLAYER_STATS_FIELDS

In [119]:
# TODO: Complete and move to starcraft_predictor processing code
ORDINAL_UPGRADES = {
    "Protoss": {},
    "Terran": {
        "ground_armour": [f"TerranInfantryArmorsLevel{i}" for i in [1, 2, 3]],
    },
    "Zerg": {
        "melee_attack": [f"ZergMeleeWeaponsLevel{i}" for i in [1, 2, 3]],
        "ranged_attack": [f"ZergMissileWeaponsLevel{i}" for i in [1, 2, 3]],
        "ground_armour": [f"ZergGroundArmorsLevel{i}" for i in [1, 2, 3]],
        "air_attack": [f"ZergFlyerWeaponsLevel{i}" for i in [1, 2, 3]],
        "air_armour": [f"ZergFlyerArmorsLevel{i}" for i in [1, 2, 3]],
    },
}

### Helper functions

In [38]:
def create_tabbed_plots(plot_functions, titles, overall_title=None):
    """
    Create a tabbed interface with multiple plots.
    
    Args:
        plot_functions (list): List of functions that create plots
        titles (list): List of titles for each tab
        overall_title (str, optional): Title for the entire tabbed interface
    """
    # Create tabs
    tab = widgets.Tab()
    tab.children = [widgets.Output() for _ in plot_functions]
    
    # Set tab titles
    for i, title in enumerate(titles):
        tab.set_title(i, title)
    
    # Create plots in each tab
    for i, plot_func in enumerate(plot_functions):
        with tab.children[i]:
            plot_func()
            plt.show()
    
    # Create container for title and tabs
    container = widgets.VBox()
    
    # Add title if provided
    if overall_title:
        title_widget = widgets.HTML(value=f"<h2>{overall_title}</h2>")
        container.children = [title_widget, tab]
    else:
        container.children = [tab]
    
    # Display the tabbed interface
    display(container)


In [51]:
def plot_shared_features_over_time(match, shared_features):
    """Plot a selection of shared features for both players over time."""
    feature_plots = list()

    player_1_race = matchup_data["player_1_race"].iloc[0]
    player_2_race = matchup_data["player_2_race"].iloc[0]
    
    all_features = []
    for feature in shared_features:
        all_features += [f"player_{x}_{feature}" for x in [1, 2]]

    grouped_match = match.groupby("seconds")[all_features].mean()
    count = match.groupby("seconds")[["filehash"]].count()
    count.rename(columns={"filehash": "count"}, inplace=True)

    for feature in shared_features:
        def plot_function(feature=feature):
            fig, ax = plt.subplots(figsize=(10, 5))

            sns.lineplot(
                x="seconds",
                y="count",
                data=count,
                ax=ax,
                color="grey",
                alpha=0.3,
            )

            ax2 = ax.twinx()

            sns.lineplot(
                x="seconds",
                y=f"player_1_{feature}",
                color="blue",
                data=grouped_match,
                label=f"player_1_{player_1_race}",
                ax=ax2,
            )
            sns.lineplot(
                x="seconds",
                y=f"player_2_{feature}",
                color="orange",
                data=grouped_match,
                label=f"player_2_{player_2_race}",
                ax=ax2,
            )    
            
            plt.title(f"{feature} over time")
            plt.xlabel("Time (seconds)")
            plt.ylabel(feature)
            plt.legend()
        feature_plots.append(plot_function)

    create_tabbed_plots(feature_plots, shared_features, overall_title="Average feature value by seconds")

In [52]:
def plot_shared_feature_distribution(matchup_data, shared_features):
    """Plot shared feature distributions."""
    feature_plots = list()

    player_1_race = matchup_data["player_1_race"].iloc[0]
    player_2_race = matchup_data["player_2_race"].iloc[0]
    
    for feature in shared_features:
        def plot_function(feature=feature):
            plt.figure(figsize=(10, 5))
            sns.histplot(matchup_data[f"player_1_{feature}"], color="blue", label=player_1_race)
            sns.histplot(matchup_data[f"player_2_{feature}"], color="orange", label=player_2_race)
            plt.title(f"{feature} histogram by race")
            plt.legend()
        feature_plots.append(plot_function)

    create_tabbed_plots(feature_plots, shared_features, overall_title=f"Winner: {match['winner'].unique()[0]}")

In [53]:
def plot_player_features_over_time(match, unit_features, player_number):
    """Plot individual player features over time, such as unit counts, etc."""
    feature_plots = list()

    all_features = [f"player_{player_number}_{feature}" for feature in unit_features]
    match_grouped = match.groupby("seconds")[all_features].mean()

    for feature in unit_features:
        def plot_function(feature=feature):
            plt.figure(figsize=(10, 5))
            sns.lineplot(
                x="seconds",
                y=f"player_{player_number}_{feature}",
                color="blue",
                data=match_grouped,
                label=f"player_{player_number}",
            )
            plt.title(f"{feature} over time")
            plt.xlabel("Time (seconds)")
            plt.ylabel(feature)
            plt.legend()
            plt.show()
        feature_plots.append(plot_function)

    create_tabbed_plots(feature_plots, unit_features)

In [122]:
def plot_upgrade_comparison(match):
    """Plot upgrade comparisons for all overlapping upgrades, eg: melee_weapons."""
    player_1_race = matchup_data["player_1_race"].iloc[0]
    player_2_race = matchup_data["player_2_race"].iloc[0]

    cross_over_upgrades = [
        x for x in ORDINAL_UPGRADES[player_1_race].keys()
        if x in ORDINAL_UPGRADES[player_2_race].keys()
    ]

    print(cross_over_upgrades)
    match_copy = match[["filehash", "seconds"]].copy()

    for upgrade in cross_over_upgrades:
        player_1_features = [f"player_1_{x}" for x in ORDINAL_UPGRADES[player_1_race][upgrade]]
        player_2_features = [f"player_2_{x}" for x in ORDINAL_UPGRADES[player_2_race][upgrade]]
    
        match_copy[f"player_1_{upgrade}"] = match[player_1_features].sum(axis=1)
        match_copy[f"player_2_{upgrade}"] = match[player_2_features].sum(axis=1)

    plot_shared_features_over_time(match_copy, cross_over_upgrades)

### Load Data

In [54]:
data = {
    # "Protoss vs Protoss": pd.read_pickle("data/Protoss_vs_Protoss.pkl"),
    # "Zerg vs Zerg": pd.read_pickle("data/Zerg_vs_Zerg.pkl"),
    # "Terran vs Terran": pd.read_pickle("data/Terran_vs_Terran.pkl"),
    # "Protoss vs Zerg": pd.read_pickle("data/Protoss_vs_Zerg.pkl"),
    "Terran vs Zerg": pd.read_pickle("../../data/Terran_vs_Zerg.pkl"),
    # "Protoss vs Terran": pd.read_pickle("data/Protoss_vs_Terran.pkl"),
}

## High Level Analysis
Look at some general statistics about each matchup, before diving into specific matchup features

In [7]:
for matchup, data_sample in data.items():
    print(matchup)
    print("number of games: ", data_sample["filehash"].nunique())
    print("number of features: ", data_sample.shape[1])
    print("average target: ", data_sample[data_sample["seconds"] == 0]["winner"].mean())
    print("\n")


Terran vs Zerg
number of games:  302
number of features:  132
average target:  1.4602649006622517




All 'average target' values are ~1.5, which is what we would expect (the target right now is 1 or 2, not 0 or 1).

### Check for broken features
Here we check for broken features. We check for any features which:
- Never contain a non-zero count across all games

This is to ensure that the feature extraction process during the replay ingestion is not broken. We should expect to see all units being built at least once, and we should expect units to die throughout games.

In [8]:
# generate the full dataset, which is a concatination of all the sub datasets
full_data = pd.concat(list(data.values()), ignore_index=True)
full_data.fillna(0, inplace=True)

In [9]:
for race in ["Protoss", "Terran", "Zerg"]:
    for unit in TRACKED_UNIT_TYPES[race]:
        features = [x for x in [f"player_1_{unit}", f"player_2_{unit}"] if x in full_data.columns]
        for feature in features:
            if full_data[feature].max() == 0:
                print(feature)
        

**Conclusion:** Only Ultralisk and BroodLord and missing for `player_1`. Due to our matchups being ordered alphabetically, `player_1` Zerg only happens in ZvZ mirror matchups. It is reasonable to assume that in all of our replays, no Ultralisks or BroodLords were made during ZvZ.

In [10]:
# generate all unit features in the full_data
all_unit_features = []
for race in ["Protoss", "Terran", "Zerg"]:
    for unit in TRACKED_UNIT_TYPES[race]:
        all_unit_features += [x for x in [f"player_1_{unit}", f"player_2_{unit}"] if x in full_data.columns]

In [11]:
# create a data subset with all features shifted by 1. This allows us to calcualte the change in feature.
# We should see some negative changes in features, meaning units are dying!
data_subset = full_data[["seconds"] + all_unit_features].copy()
for feature in all_unit_features:
    data_subset[feature] = data_subset[feature] - data_subset[feature].shift(1)

# remove seconds == 0 which will be the handover row between games where the shift logic makes no sense
data_subset = data_subset[data_subset["seconds"] == 0]

In [12]:
for feature in all_unit_features:
    if data_subset[feature].min() >= 0:
        print(feature)
        print(data_subset[feature].max())

Again, the only non-decreasing units are zerg units in ZvZ matchups. We can assume this is accurate and that our data loading is correct.

## Matchup Analysis

### Mirror Matchup

In [12]:
matchup = "Zerg vs Zerg"

In [40]:
filehash = data[matchup]["filehash"].unique()[0]
race = matchup.split(" ")[0]

In [41]:
# plot player stats fields over time
plot_shared_features_over_time(data[matchup], PLAYER_STATS_FIELDS, filehash)

In [43]:
# can plot upgrade features as shared features for mirror matchups
plot_shared_features_over_time(data[matchup], TRACKED_UNIT_TYPES[race], filehash)

In [42]:
# can plot upgrade features as shared features for mirror matchups
plot_shared_features_over_time(data[matchup], TRACKED_UPGRADE_TYPES[race], filehash)

### Non-mirror Matchup

In [128]:
matchup = "Terran vs Zerg"
matchup_data = data[matchup]

In [132]:
# select a filehash from the matchup to explore
filehash = "a91828f3d07132cd3982c58ed6fccabca5a9ae1fe25bd5d8325382eca8df9623"

race_1 = matchup.split(" ")[0]
race_2 = matchup.split(" ")[-1]

match = matchup_data[matchup_data["filehash"] == filehash]

#### Shared Features
Explore the shared features for the matchup. This includes non race specific features like resources spent, army size, etc.

In [133]:
# plot average player stats fields over time for all games in the matchup, split by player
# plot_shared_features_over_time(matchup_data, PLAYER_STATS_FIELDS)

# plot player stats fields over time for a specific game, split by player
plot_shared_features_over_time(match, PLAYER_STATS_FIELDS)

In [58]:
# plot the histogram for each feature over all games in the matchup, split by player
plot_shared_feature_distribution(matchup_data, PLAYER_STATS_FIELDS)

#### Player Features
Analyse the player specific features, which includes both the unit and upgrade counts for each player. For non-mirror matchups these counts are plotted separately for each player, as there is no crossover of unit types.

In [120]:
# plot average race 1 unit counts over time
plot_player_features_over_time(matchup_data, TRACKED_UNIT_TYPES[race_1], 1)

In [121]:
# plot average race 2 unit counts over time
plot_player_features_over_time(matchup_data, TRACKED_UNIT_TYPES[race_2], 2)

In [61]:
# plot average race 1 upgrades over time
plot_player_features_over_time(matchup_data, TRACKED_UPGRADE_TYPES[race_1], 1)

KeyError: "Columns not found: 'player_1_InterferenceMatrix', 'player_1_TerranInfantryWeaponsLevel1'"

In [62]:
# plot average race 2 upgrades over time
plot_player_features_over_time(matchup_data, TRACKED_UPGRADE_TYPES[race_2], 2)

#### Upgrade Comparisons
Although race upgrades are different, they can often be compared. Here a standard naming convention for upgrades is adopted across all races, and the upgrade times are evaluated.

In [134]:
plot_upgrade_comparison(matchup_data)

['ground_armour']
